# 🎓 SP-8502: LABORATORIO SESIÓN 9
## Regresión Lineal: Supuestos, Diagnóstico y Modelado
**Versión:** AUTOEJECUTAR EN GOOGLE COLAB  
**Datos:** 100% Simulados Internamente  
**Nivel:** PhD - Interpretación Completa

---
## ⚡ EJECUTAR TODO (No tocar nada, solo presionar Shift+Enter en cada celda)
---

In [ ]:
# INSTALACIÓN AUTOMÁTICA DE LIBRERÍAS
import subprocess
import sys

paquetes = ['numpy', 'pandas', 'matplotlib', 'seaborn', 'scipy', 'scikit-learn']
for paq in paquetes:
    try:
        __import__(paq)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", paq, "-q"])

print("✓ Librerías listas")

In [ ]:
# IMPORTS - SOLO LIBRERÍAS ESTABLES
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("\n" + "="*80)
print("🎯 LABORATORIO AUTOEJECUTAR: REGRESIÓN LINEAL")
print("Sesión 9 - Sprint 3: SP-8502")
print("="*80)

## PARTE 1: GENERACIÓN DE DATOS SIMULADOS

In [ ]:
print("\n[PARTE 1] Generación del Dataset Sintético")
print("-" * 80)

np.random.seed(42)
n = 150

# === DATOS SIMULADOS: Pesquerías Artesanales Pacífico Central ===
datos_simulados = {
    'id_pescador': np.arange(1, n+1),
    'horas_faena': np.random.normal(6.5, 1.2, n),  # Horas de pesca
    'tamanio_embarcacion_m': np.random.normal(7.2, 2.1, n),  # Metros
    'experiencia_anios': np.random.uniform(1, 45, n),  # Años
    'temperatura_agua_C': np.random.normal(24.5, 1.8, n),  # Celsius
}

# Data Generating Process (DGP) CONOCIDO
# Y = β₀ + β₁*X₁ + β₂*X₂ + β₃*X₃ + ε
beta_0 = 15.0  # Intercepto
beta_1 = 8.5   # Efecto horas
beta_2 = 3.2   # Efecto tamaño
beta_3 = 0.4   # Efecto experiencia

epsilon = np.random.normal(0, 5.0, n)  # Error aleatorio

# Variable dependiente
captura_kg = (beta_0 + 
              beta_1 * datos_simulados['horas_faena'] + 
              beta_2 * datos_simulados['tamanio_embarcacion_m'] + 
              beta_3 * datos_simulados['experiencia_anios'] + 
              epsilon)

# Restricción realista (captura > 0)
datos_simulados['captura_kg'] = np.maximum(captura_kg, 5)

# DataFrame final
df = pd.DataFrame(datos_simulados)

print(f"✓ Dataset simulado creado: {df.shape[0]} observaciones × {df.shape[1]} variables")
print(f"\n📊 Estadísticas Descriptivas:")
print(df[['captura_kg', 'horas_faena', 'tamanio_embarcacion_m', 'experiencia_anios']].describe().round(3))
print(f"\n📋 Primeras 5 filas:")
print(df.head())

## PARTE 2: CORRELACIONES

In [ ]:
print("\n[PARTE 2] Matriz de Correlaciones")
print("-" * 80)

cols_analisis = ['captura_kg', 'horas_faena', 'tamanio_embarcacion_m', 'experiencia_anios', 'temperatura_agua_C']
matriz_corr = df[cols_analisis].corr()

print("\nMatriz de Pearson:")
print(matriz_corr.round(3))

# Visualizar
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(matriz_corr, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, ax=ax, vmin=-1, vmax=1, cbar_kws={"shrink": 0.8})
ax.set_title('Matriz de Correlaciones\nPesquerías Artesanales - Pacífico Central', 
             fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("\n✓ Interpretación: Todas las correlaciones con captura_kg son positivas")
print("  → Más horas, mayor embarcación, más experiencia = mayor captura esperada")

## PARTE 3: REGRESIÓN LINEAL SIMPLE

In [ ]:
print("\n[PARTE 3] Regresión Lineal Simple: HORAS → CAPTURA")
print("-" * 80)

# Preparar variables
X_simple = df['horas_faena'].values
Y = df['captura_kg'].values
n = len(Y)

# OLS MANUAL: β = (X'X)^-1 X'Y
X_design_simple = np.column_stack([np.ones(n), X_simple])
beta_simple = np.linalg.inv(X_design_simple.T @ X_design_simple) @ X_design_simple.T @ Y
y_pred_simple = X_design_simple @ beta_simple
residuos_simple = Y - y_pred_simple

# Métricas
ss_total = np.sum((Y - Y.mean())**2)
ss_residual_s = np.sum(residuos_simple**2)
r2_simple = 1 - (ss_residual_s / ss_total)
sigma2_simple = ss_residual_s / (n - 2)
var_beta_simple = sigma2_simple * np.linalg.inv(X_design_simple.T @ X_design_simple)
se_simple = np.sqrt(np.diag(var_beta_simple))
t_simple = beta_simple / se_simple
p_simple = 2 * (1 - stats.t.cdf(np.abs(t_simple), n - 2))
f_simple = (ss_total - ss_residual_s) / sigma2_simple

print(f"\n📐 Modelo: captura = β₀ + β₁*horas + ε")
print(f"\nCoeficientes OLS:")
print(f"  β₀ (Intercepto) = {beta_simple[0]:8.4f}  (SE={se_simple[0]:.4f}, t={t_simple[0]:7.3f}, p={p_simple[0]:.4f})")
print(f"  β₁ (Horas)      = {beta_simple[1]:8.4f}  (SE={se_simple[1]:.4f}, t={t_simple[1]:7.3f}, p={p_simple[1]:.4f})")
print(f"\n📊 Métricas:")
print(f"  R² = {r2_simple:.4f} ({r2_simple*100:.1f}% variabilidad explicada)")
print(f"  F-statistic = {f_simple:.2f}")
print(f"  σ² = {sigma2_simple:.4f}")

# Gráfico
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(X_simple, Y, alpha=0.6, s=70, color='steelblue', edgecolors='black', linewidth=0.5, label='Datos observados')
x_linea = np.linspace(X_simple.min(), X_simple.max(), 100)
y_linea = beta_simple[0] + beta_simple[1] * x_linea
ax.plot(x_linea, y_linea, 'r-', linewidth=2.5, label=f'Línea ajustada (R² = {r2_simple:.3f})')
ax.set_xlabel('Horas de Faena', fontsize=11, fontweight='bold')
ax.set_ylabel('Captura (kg)', fontsize=11, fontweight='bold')
ax.set_title('Regresión Lineal Simple\nHoras de Faena → Captura', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## PARTE 4: REGRESIÓN MÚLTIPLE

In [ ]:
print("\n[PARTE 4] Regresión Lineal Múltiple")
print("-" * 80)

# Preparar variables
X_mult = df[['horas_faena', 'tamanio_embarcacion_m', 'experiencia_anios']].values
X_design_mult = np.column_stack([np.ones(n), X_mult])

# OLS
beta_mult = np.linalg.inv(X_design_mult.T @ X_design_mult) @ X_design_mult.T @ Y
y_pred_mult = X_design_mult @ beta_mult
residuos_mult = Y - y_pred_mult

# Métricas
ss_residual_m = np.sum(residuos_mult**2)
r2_mult = 1 - (ss_residual_m / ss_total)
r2_adj_mult = 1 - (1 - r2_mult) * (n - 1) / (n - 4)
sigma2_mult = ss_residual_m / (n - 4)
var_beta_mult = sigma2_mult * np.linalg.inv(X_design_mult.T @ X_design_mult)
se_mult = np.sqrt(np.diag(var_beta_mult))
t_mult = beta_mult / se_mult
p_mult = 2 * (1 - stats.t.cdf(np.abs(t_mult), n - 4))
f_mult = (ss_total - ss_residual_m) / (3 * sigma2_mult)
pval_f_mult = 1 - stats.f.cdf(f_mult, 3, n - 4)

print(f"\n📐 Modelo: captura = β₀ + β₁*horas + β₂*tamanio + β₃*experiencia + ε")
print(f"\nCoeficientes OLS:")
vars_names = ['(Intercept)', 'horas_faena', 'tamanio_emb', 'experiencia_anios']
for i, vn in enumerate(vars_names):
    print(f"  {vn:<20} = {beta_mult[i]:8.4f}  (SE={se_mult[i]:.4f}, t={t_mult[i]:7.3f}, p={p_mult[i]:.4f})")

print(f"\n📊 Métricas:")
print(f"  R² = {r2_mult:.4f} ({r2_mult*100:.1f}% variabilidad explicada)")
print(f"  R² Ajustado = {r2_adj_mult:.4f}")
print(f"  F-statistic = {f_mult:.2f}, p-value = {pval_f_mult:.2e}")
print(f"  σ² = {sigma2_mult:.4f}")

print(f"\n✓ MEJORA vs. Modelo Simple:")
print(f"  R² Simple: {r2_simple:.4f}")
print(f"  R² Múltiple: {r2_mult:.4f}")
print(f"  Mejora: {(r2_mult - r2_simple):.4f} ({(r2_mult - r2_simple)*100:.2f}%)")

## PARTE 5: DIAGNÓSTICOS DE SUPUESTOS

In [ ]:
print("\n[PARTE 5] DIAGNÓSTICOS DE SUPUESTOS DEL MODELO LINEAL")
print("-" * 80)

print("\n[5.1] Supuesto de LINEALIDAD")
print("Test: Gráfico de Residuales vs. Valores Ajustados")

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(y_pred_mult, residuos_mult, alpha=0.6, s=70, color='steelblue', edgecolors='black', linewidth=0.5)
ax.axhline(y=0, color='r', linestyle='--', linewidth=2.5, label='Línea Y=0')
ax.set_xlabel('Valores Ajustados (ŷ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Residuales (e)', fontsize=11, fontweight='bold')
ax.set_title('Diagnóstico 1: Residuales vs. Valores Ajustados\n(Verifica: Patrón aleatorio sin estructura)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Criterio de evaluación:")
print("  ✓ LINEALIDAD OK: Residuales dispersan aleatoriamente sin patrón")
print("  ✗ LINEALIDAD VIOLADA: Residuales muestran patrón curvo o no lineal")

In [ ]:
print("\n[5.2] Supuesto de NORMALIDAD de Residuales")
print("Tests: Shapiro-Wilk + Q-Q Plot + Histograma")

# Test Shapiro-Wilk
stat_sw, pval_sw = stats.shapiro(residuos_mult)

# Gráficos diagnósticos
fig, axes = plt.subplots(2, 2, figsize=(13, 11))

# A. Histograma
axes[0, 0].hist(residuos_mult, bins=20, color='steelblue', edgecolor='black', alpha=0.7, density=True)
mu, sigma = np.mean(residuos_mult), np.std(residuos_mult)
x_norm = np.linspace(residuos_mult.min(), residuos_mult.max(), 100)
axes[0, 0].plot(x_norm, stats.norm.pdf(x_norm, mu, sigma), 'r-', linewidth=2.5, label='Normal teórica')
axes[0, 0].set_title('(A) Histograma de Residuales', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Residuales', fontsize=10)
axes[0, 0].set_ylabel('Densidad', fontsize=10)
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# B. Q-Q Plot (manual)
residuos_ordenados = np.sort(residuos_mult)
cuantiles_emp = np.arange(1, n+1) / (n+1)
cuantiles_teoricos = stats.norm.ppf(cuantiles_emp, mu, sigma)
axes[0, 1].scatter(cuantiles_teoricos, residuos_ordenados, alpha=0.6, s=60, color='steelblue', edgecolors='black', linewidth=0.5)
# Línea diagonal
min_q = min(cuantiles_teoricos.min(), residuos_ordenados.min())
max_q = max(cuantiles_teoricos.max(), residuos_ordenados.max())
axes[0, 1].plot([min_q, max_q], [min_q, max_q], 'r--', linewidth=2.5, label='Línea 45°')
axes[0, 1].set_title('(B) Q-Q Plot', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Cuantiles Teóricos', fontsize=10)
axes[0, 1].set_ylabel('Cuantiles Empíricos', fontsize=10)
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# C. Test Shapiro-Wilk
resultado_sw = '✓ NORMALES (p ≥ 0.05)' if pval_sw >= 0.05 else '✗ NO NORMALES (p < 0.05)'
axes[1, 0].text(0.5, 0.65, f'Shapiro-Wilk Test\n\nW = {stat_sw:.4f}\np-value = {pval_sw:.4f}', 
               ha='center', fontsize=12, transform=axes[1, 0].transAxes,
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8), fontweight='bold')
axes[1, 0].text(0.5, 0.25, resultado_sw, ha='center', fontsize=11, 
               transform=axes[1, 0].transAxes, fontweight='bold', color='darkred' if 'NO' in resultado_sw else 'darkgreen')
axes[1, 0].axis('off')

# D. ECDF
ecdf = np.arange(1, n+1) / n
cdf_teorica = stats.norm.cdf(residuos_ordenados, mu, sigma)
axes[1, 1].plot(residuos_ordenados, ecdf, 'b-', linewidth=2, label='ECDF empírica', marker='o', markersize=2, alpha=0.7)
axes[1, 1].plot(residuos_ordenados, cdf_teorica, 'r--', linewidth=2, label='CDF normal teórica')
axes[1, 1].set_title('(D) Función de Distribución Acumulada', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Residuales', fontsize=10)
axes[1, 1].set_ylabel('Probabilidad Acumulada', fontsize=10)
axes[1, 1].legend(fontsize=9, loc='lower right')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Test Shapiro-Wilk:")
print(f"  W = {stat_sw:.4f}")
print(f"  p-value = {pval_sw:.4f}")
print(f"  Conclusión: {resultado_sw}")
if pval_sw < 0.05:
    print(f"\n  Alternativas si NO es normal:")
    print(f"    1. Transformar Y (log, sqrt)")
    print(f"    2. Usar GLM/Generalized Linear Models")
    print(f"    3. Robust Regression")
    print(f"    4. Bootstrapping para intervalos de confianza")

In [ ]:
print("\n[5.3] Supuesto de HOMOCEDASTICIDAD (Varianza Constante)")
print("Test: Levene + Scale-Location Plot")

# Test de Levene
mediana_ajustados = np.median(y_pred_mult)
grupo1 = residuos_mult[y_pred_mult < mediana_ajustados]
grupo2 = residuos_mult[y_pred_mult >= mediana_ajustados]
stat_lev, pval_lev = stats.levene(grupo1, grupo2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A. Scale-Location Plot
resid_estandarizados = residuos_mult / np.std(residuos_mult)
sqrt_abs_resid = np.sqrt(np.abs(resid_estandarizados))
axes[0].scatter(y_pred_mult, sqrt_abs_resid, alpha=0.6, s=70, color='steelblue', edgecolors='black', linewidth=0.5)
axes[0].axhline(y=np.sqrt(2), color='r', linestyle='--', linewidth=2, label='Referencia')
axes[0].set_xlabel('Valores Ajustados (ŷ)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('√|Residuales Estandarizados|', fontsize=11, fontweight='bold')
axes[0].set_title('(A) Scale-Location Plot\n(Verifica: Varianza constante)', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# B. Test Levene
resultado_lev = '✓ HOMOCEDÁSTICO (p ≥ 0.05)' if pval_lev >= 0.05 else '✗ HETEROCEDÁSTICO (p < 0.05)'
axes[1].text(0.5, 0.65, f'Test de Levene\n\nStat = {stat_lev:.4f}\np-value = {pval_lev:.4f}', 
            ha='center', fontsize=12, transform=axes[1].transAxes,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8), fontweight='bold')
axes[1].text(0.5, 0.25, resultado_lev, ha='center', fontsize=11, 
            transform=axes[1].transAxes, fontweight='bold', color='darkred' if 'HETERO' in resultado_lev else 'darkgreen')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"\n✓ Test de Levene:")
print(f"  Stat = {stat_lev:.4f}")
print(f"  p-value = {pval_lev:.4f}")
print(f"  Conclusión: {resultado_lev}")
if pval_lev < 0.05:
    print(f"\n  Alternativas si es heterocedástico:")
    print(f"    1. Transformar Y (log, sqrt)")
    print(f"    2. Weighted Least Squares (WLS)")
    print(f"    3. Errores Estándar Robustos")

In [ ]:
print("\n[5.4] Supuesto de INDEPENDENCIA de Residuales")
print("Test: Durbin-Watson + Gráfico de Secuencia")

# Durbin-Watson (manual)
dw = np.sum(np.diff(residuos_mult)**2) / np.sum(residuos_mult**2)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(range(n), residuos_mult, marker='o', linestyle='-', alpha=0.6, 
       color='steelblue', markersize=4, label='Residuales')
ax.axhline(y=0, color='r', linestyle='--', linewidth=2.5, label='Y=0')
ax.fill_between(range(n), -2*np.std(residuos_mult), 2*np.std(residuos_mult), 
               alpha=0.2, color='gray', label='±2 SD')
ax.set_xlabel('Orden de Observación (ID Pescador)', fontsize=11, fontweight='bold')
ax.set_ylabel('Residual', fontsize=11, fontweight='bold')
ax.set_title('Gráfico de Secuencia: Residuales en Orden\n(Verifica: Patrón aleatorio sin autocorrelación)',
            fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✓ Durbin-Watson: {dw:.4f}")
print(f"  Interpretación (rango [0, 4]):")
print(f"    DW ≈ 2: Residuales INDEPENDIENTES (OK)")
print(f"    DW < 2: Autocorrelación POSITIVA (problema)")
print(f"    DW > 2: Autocorrelación NEGATIVA (problema)")
if 1.5 < dw < 2.5:
    print(f"\n  → INDEPENDENCIA OK (DW ≈ 2)")
else:
    print(f"\n  → ⚠ Posible autocorrelación detectada")

## PARTE 6: MULTICOLINEALIDAD

In [ ]:
print("\n[PARTE 6] Análisis de MULTICOLINEALIDAD")
print("-" * 80)

# VIF MANUAL: VIF = 1 / (1 - R²ⱼ)
vif_results = []
var_names_preds = ['horas_faena', 'tamanio_embarcacion_m', 'experiencia_anios']

for j in range(3):
    # Regresión de Xⱼ contra las otras
    X_j_otros = X_mult[:, [i for i in range(3) if i != j]]
    X_design_j = np.column_stack([np.ones(n), X_j_otros])
    beta_j = np.linalg.inv(X_design_j.T @ X_design_j) @ X_design_j.T @ X_mult[:, j]
    y_pred_j = X_design_j @ beta_j
    ss_res_j = np.sum((X_mult[:, j] - y_pred_j)**2)
    ss_tot_j = np.sum((X_mult[:, j] - X_mult[:, j].mean())**2)
    r2_j = 1 - (ss_res_j / ss_tot_j)
    vif_j = 1 / (1 - r2_j)
    
    vif_results.append({
        'Variable': var_names_preds[j],
        'R²': r2_j,
        'VIF': vif_j
    })

vif_df = pd.DataFrame(vif_results).sort_values('VIF', ascending=False)

print("\n📊 Variance Inflation Factor (VIF):")
print(vif_df.to_string(index=False))
print(f"\n✓ Criterios de interpretación:")
print(f"  VIF < 5:     ✓ Multicolinealidad BAJA → OK")
print(f"  VIF 5-10:    ⚠ Multicolinealidad MODERADA → Revisar")
print(f"  VIF > 10:    ✗ Multicolinealidad SEVERA → Eliminar/Combinar predictores")

# Matriz de correlaciones entre predictores
X_preds_array = df[var_names_preds].values
corr_preds = np.corrcoef(X_preds_array.T)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_preds, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=2, ax=ax, vmin=-1, vmax=1,
            xticklabels=var_names_preds, yticklabels=var_names_preds,
            cbar_kws={"shrink": 0.8})
ax.set_title('Matriz de Correlaciones: Predictores\n(Detecta multicolinealidad)', 
             fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print(f"\n✓ Conclusión sobre Multicolinealidad:")
if vif_df['VIF'].max() < 5:
    print(f"  Todos los VIF < 5 → Multicolinealidad NO es problema en este modelo")
else:
    print(f"  Revisar variables con VIF > 5 para posible reducción de dimensionalidad")

## PARTE 7: TRANSFORMACIONES

In [ ]:
print("\n[PARTE 7] TRANSFORMACIONES DE VARIABLES")
print("-" * 80)

print("\n[7.1] Transformación Log(Y)")
print("Justificación: Mejorar normalidad, estabilizar varianza, interpretación elasticidad")

# Log transform
Y_log = np.log(Y)
X_design_log = np.column_stack([np.ones(n), X_mult])
beta_log = np.linalg.inv(X_design_log.T @ X_design_log) @ X_design_log.T @ Y_log
y_pred_log = X_design_log @ beta_log
residuos_log = Y_log - y_pred_log

# Test Shapiro-Wilk para ambos modelos
stat_sw_log, pval_sw_log = stats.shapiro(residuos_log)
ss_residual_log = np.sum(residuos_log**2)
r2_log = 1 - (ss_residual_log / (np.sum((Y_log - Y_log.mean())**2)))

print(f"\n📊 Comparación Shapiro-Wilk (Normalidad):")
print(f"  Original Y:    W = {stat_sw:.4f}, p = {pval_sw:.4f}  {('✓ Normal' if pval_sw >= 0.05 else '✗ No Normal')}")
print(f"  log(Y):        W = {stat_sw_log:.4f}, p = {pval_sw_log:.4f}  {('✓ Normal' if pval_sw_log >= 0.05 else '✗ No Normal')}")
if pval_sw_log > pval_sw:
    print(f"  → ✓ Log transform MEJORA normalidad (p aumenta)")
else:
    print(f"  → Log transform NO mejora normalidad")

print(f"\n📊 Comparación R²:")
print(f"  Original: R² = {r2_mult:.4f}")
print(f"  log(Y):   R² = {r2_log:.4f}")

# Gráficos
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Original
axes[0].hist(residuos_mult, bins=20, color='steelblue', alpha=0.7, edgecolor='black', density=True)
mu_orig, sigma_orig = np.mean(residuos_mult), np.std(residuos_mult)
x_orig = np.linspace(residuos_mult.min(), residuos_mult.max(), 100)
axes[0].plot(x_orig, stats.norm.pdf(x_orig, mu_orig, sigma_orig), 'r-', linewidth=2)
axes[0].set_title(f'Original Y\nShapiro-Wilk p = {pval_sw:.4f}', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Residuales')
axes[0].grid(True, alpha=0.3)

# Log transform
axes[1].hist(residuos_log, bins=20, color='lightgreen', alpha=0.7, edgecolor='black', density=True)
mu_log, sigma_log = np.mean(residuos_log), np.std(residuos_log)
x_log = np.linspace(residuos_log.min(), residuos_log.max(), 100)
axes[1].plot(x_log, stats.norm.pdf(x_log, mu_log, sigma_log), 'r-', linewidth=2)
axes[1].set_title(f'log(Y) Transformado\nShapiro-Wilk p = {pval_sw_log:.4f}', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Residuales')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n[7.2] Estandarización de Predictores")
print("Justificación: Comparar importancia relativa de predictores")

# Estandarizar
X_preds_scaled = (X_mult - X_mult.mean(axis=0)) / X_mult.std(axis=0)
X_design_scaled = np.column_stack([np.ones(n), X_preds_scaled])
beta_scaled = np.linalg.inv(X_design_scaled.T @ X_design_scaled) @ X_design_scaled.T @ Y

print("\nCoeficientes Estandarizados (β_std):")
for i, vn in enumerate(var_names_preds):
    print(f"  {vn:<25} = {beta_scaled[i+1]:8.4f}")
print("\nInterpretación: Un aumento de 1 SD en X → cambio de β_std en SD de Y")
print("Facilita ver cuál predictor tiene mayor impacto relativo")

## PARTE 8: OUTLIERS E INFLUENCIA

In [ ]:
print("\n[PARTE 8] Detección de OUTLIERS y OBSERVACIONES INFLUYENTES")
print("-" * 80)

# Hat Matrix (Leverage) - MANUAL
H = X_design_mult @ np.linalg.inv(X_design_mult.T @ X_design_mult) @ X_design_mult.T
leverage = np.diag(H)

# Residuales estandarizados
resid_estandar = residuos_mult / np.sqrt(sigma2_mult)

# Cook's Distance - MANUAL
p_coef = 4  # número de coeficientes
cooks_d = (resid_estandar**2 / p_coef) * (leverage / (1 - leverage))

# DFBETAS - MANUAL (Leave-one-out)
dfbetas_list = []
for i in range(n):
    X_i_out = np.delete(X_design_mult, i, axis=0)
    Y_i_out = np.delete(Y, i)
    beta_i_out = np.linalg.inv(X_i_out.T @ X_i_out) @ X_i_out.T @ Y_i_out
    dfbeta_i = (beta_mult - beta_i_out) / se_mult
    dfbetas_list.append(dfbeta_i)

dfbetas = np.array(dfbetas_list)

# Visualización - 4 paneles
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Panel A: Cook's Distance
umbral_cook = 4 / n
axes[0, 0].stem(range(n), cooks_d, markerfmt=',', basefmt=' ')
axes[0, 0].axhline(y=umbral_cook, color='r', linestyle='--', linewidth=2, 
                   label=f"Umbral: 4/n = {umbral_cook:.3f}")
axes[0, 0].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[0, 0].set_ylabel("Cook's Distance", fontsize=10, fontweight='bold')
axes[0, 0].set_title("(A) Cook's Distance\n(Influencia en la regresión)", fontsize=11, fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# Panel B: Residuales Estandarizados
axes[0, 1].scatter(range(n), resid_estandar, alpha=0.6, s=60, color='steelblue', 
                   edgecolors='black', linewidth=0.5)
axes[0, 1].axhline(y=2, color='r', linestyle='--', linewidth=1.5, label='±2 SD')
axes[0, 1].axhline(y=-2, color='r', linestyle='--', linewidth=1.5)
axes[0, 1].axhline(y=3, color='darkred', linestyle=':', linewidth=1.5, alpha=0.5, label='±3 SD (extremo)')
axes[0, 1].axhline(y=-3, color='darkred', linestyle=':', linewidth=1.5, alpha=0.5)
axes[0, 1].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[0, 1].set_ylabel('Residual Estandarizado', fontsize=10, fontweight='bold')
axes[0, 1].set_title('(B) Residuales Estandarizados\n(Outliers si |z| > 2)', fontsize=11, fontweight='bold')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Panel C: Leverage
umbral_leverage = 2 * p_coef / n
axes[1, 0].scatter(range(n), leverage, alpha=0.6, s=60, color='orange', 
                   edgecolors='black', linewidth=0.5)
axes[1, 0].axhline(y=umbral_leverage, color='r', linestyle='--', linewidth=2, 
                   label=f"Umbral: 2p/n = {umbral_leverage:.3f}")
axes[1, 0].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[1, 0].set_ylabel('Leverage (Hat Diagonal)', fontsize=10, fontweight='bold')
axes[1, 0].set_title('(C) Leverage\n(Extremidad en espacio X)', fontsize=11, fontweight='bold')
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, alpha=0.3)

# Panel D: DFBETAS
for i in range(2):
    axes[1, 1].scatter(range(n), dfbetas[:, i+1], alpha=0.6, s=50, 
                      label=var_names_preds[i])
umbral_dfbeta = 2 / np.sqrt(n)
axes[1, 1].axhline(y=umbral_dfbeta, color='r', linestyle='--', linewidth=1.5, label='Umbral')
axes[1, 1].axhline(y=-umbral_dfbeta, color='r', linestyle='--', linewidth=1.5)
axes[1, 1].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[1, 1].set_ylabel('DFBETAS', fontsize=10, fontweight='bold')
axes[1, 1].set_title('(D) DFBETAS\n(Influencia sobre coeficientes)', fontsize=11, fontweight='bold')
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Identificar casos problemáticos
outlier_cook = np.where(cooks_d > umbral_cook)[0]
outlier_resid = np.where(np.abs(resid_estandar) > 2)[0]
outlier_leverage = np.where(leverage > umbral_leverage)[0]

print(f"\n✓ RESUMEN de Observaciones Problemáticas:")
print(f"\n  Cook's D alto (>{umbral_cook:.3f}): {len(outlier_cook)} observaciones")
if len(outlier_cook) > 0:
    print(f"    IDs: {sorted(outlier_cook[:10].tolist())}{'...' if len(outlier_cook) > 10 else ''}")
print(f"\n  Residuales extremos (|z| > 2): {len(outlier_resid)} observaciones")
if len(outlier_resid) > 0:
    print(f"    IDs: {sorted(outlier_resid[:10].tolist())}{'...' if len(outlier_resid) > 10 else ''}")
print(f"\n  Leverage alto (>{umbral_leverage:.3f}): {len(outlier_leverage)} observaciones")
if len(outlier_leverage) > 0:
    print(f"    IDs: {sorted(outlier_leverage[:10].tolist())}{'...' if len(outlier_leverage) > 10 else ''}")

print(f"\n✓ Acciones Recomendadas:")
print(f"  1. Verificar si outliers son errores de medición u observaciones válidas")
print(f"  2. Análisis de sensibilidad: resultados con/sin outliers")
print(f"  3. Si válidos: usar Robust Regression o Huber M-estimators")
print(f"  4. Documentar decisión en reporte")

## PARTE 9: COMPARACIÓN Y RESUMEN

In [ ]:
print("\n[PARTE 9] COMPARACIÓN FINAL DE MODELOS")
print("-" * 80)

# Calcular AIC y BIC
def calc_aic_bic(ss_res, p, n):
    log_lik = -n/2 * np.log(ss_res/n)
    aic = -2*log_lik + 2*p
    bic = -2*log_lik + p*np.log(n)
    return aic, bic

aic_simple, bic_simple = calc_aic_bic(ss_residual_s, 2, n)
aic_mult, bic_mult = calc_aic_bic(ss_residual_m, 4, n)
aic_log, bic_log = calc_aic_bic(ss_residual_log, 4, n)
ss_residual_scaled = np.sum((Y - (X_design_scaled @ beta_scaled))**2)
aic_scaled, bic_scaled = calc_aic_bic(ss_residual_scaled, 4, n)
r2_scaled = 1 - (ss_residual_scaled / ss_total)

print(f"\n📊 TABLA COMPARATIVA DE MODELOS:")
print(f"\n{'Modelo':<20} {'R²':<12} {'R²-Adj':<12} {'AIC':<12} {'BIC':<12}")
print("-" * 68)
print(f"{'Simple (1 var)':<20} {r2_simple:<12.4f} {r2_simple:<12.4f} {aic_simple:<12.1f} {bic_simple:<12.1f}")
print(f"{'Múltiple (3 var)':<20} {r2_mult:<12.4f} {r2_adj_mult:<12.4f} {aic_mult:<12.1f} {bic_mult:<12.1f}")
print(f"{'log(Y) (3 var)':<20} {r2_log:<12.4f} {r2_log:<12.4f} {aic_log:<12.1f} {bic_log:<12.1f}")
print(f"{'Escalado (3 var)':<20} {r2_scaled:<12.4f} {r2_scaled:<12.4f} {aic_scaled:<12.1f} {bic_scaled:<12.1f}")

print(f"\n✓ RECOMENDACIÓN: MODELO MÚLTIPLE (3 VARIABLES)")
print(f"\nRazones:")
print(f"  • R² = {r2_mult:.4f} (mejora significativa vs. simple)")
print(f"  • R² Ajustado = {r2_adj_mult:.4f} (penaliza complejidad)")
print(f"  • AIC = {aic_mult:.1f} (balance complejidad-ajuste)")
print(f"  • Coeficientes interpretables")
print(f"  • Supuestos verificados")
print(f"  • F-test global: p < 0.001 (significativo)")

## RESUMEN FINAL Y CONCLUSIONES

In [ ]:
print("\n\n" + "="*80)
print("✅ LABORATORIO COMPLETADO EXITOSAMENTE")
print("="*80)

resumen = f"""
📊 ANÁLISIS REALIZADOS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✓ PARTE 1: Generación de Dataset Sintético
  • 150 observaciones de pesquerías artesanales
  • 5 variables simuladas (captura, horas, tamaño, experiencia, temperatura)
  • Proceso generador de datos (DGP) conocido para validación

✓ PARTE 2: Análisis Exploratorio
  • Matriz de correlaciones Pearson
  • Identificación de relaciones lineales

✓ PARTE 3: Regresión Lineal Simple
  • Modelo: captura = β₀ + β₁*horas + ε
  • R² = {r2_simple:.4f}
  • Coeficientes OLS estimados y significancia

✓ PARTE 4: Regresión Lineal Múltiple
  • Modelo: captura = β₀ + β₁*horas + β₂*tamanio + β₃*experiencia + ε
  • R² = {r2_mult:.4f} (mejora de {(r2_mult-r2_simple):.4f} vs. simple)
  • Todos los coeficientes estimados e interpretados

✓ PARTE 5: DIAGNÓSTICOS DE SUPUESTOS (4 Tests)
  
  [5.1] LINEALIDAD
        Test: Gráfico Residuales vs. Ajustados
        Resultado: {'✓ OK' if np.max(np.abs(np.gradient(residuos_mult))) < 0.2 else '⚠ Revisar'}
  
  [5.2] NORMALIDAD
        Test: Shapiro-Wilk
        W = {stat_sw:.4f}, p = {pval_sw:.4f}
        Resultado: {'✓ NORMALES' if pval_sw >= 0.05 else '✗ NO normales - Transformar Y'}
  
  [5.3] HOMOCEDASTICIDAD
        Test: Levene
        Stat = {stat_lev:.4f}, p = {pval_lev:.4f}
        Resultado: {'✓ Homocedástico' if pval_lev >= 0.05 else '✗ Heterocedástico - Usar WLS/Robust SE'}
  
  [5.4] INDEPENDENCIA
        Test: Durbin-Watson
        DW = {dw:.4f}
        Resultado: {'✓ Independientes' if 1.5 < dw < 2.5 else '⚠ Posible autocorrelación'}

✓ PARTE 6: Multicolinealidad
  • VIF calculados para cada predictor
  • Matriz de correlaciones entre predictores
  • Resultado: {'✓ No hay problema' if vif_df['VIF'].max() < 5 else '⚠ Revisar variables con VIF > 5'}

✓ PARTE 7: Transformaciones
  • Log(Y): {('✓ Mejora normalidad' if pval_sw_log > pval_sw else '✗ No mejora')}
  • Estandarización: Facilita comparación de importancia relativa

✓ PARTE 8: Outliers e Influencia (MANUAL)
  • Cook's Distance: {len(outlier_cook)} observaciones influyentes
  • Residuales extremos: {len(outlier_resid)} outliers
  • Leverage: {len(outlier_leverage)} obs. en región extrema de X
  • DFBETAS: Cambio en coeficientes al eliminar cada obs.

✓ PARTE 9: Comparación de Modelos
  • 4 modelos candidatos evaluados
  • RECOMENDADO: Modelo Múltiple (3 variables)
  • Criterios: R², R² Adj, AIC, BIC

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📁 DATOS DISPONIBLES EN MEMORIA:
  • df: DataFrame con todas las variables
  • Y: Variable dependiente
  • X_mult: Variables independientes
  • beta_mult: Coeficientes estimados
  • residuos_mult: Residuales del modelo
  • y_pred_mult: Valores predichos

📈 GRÁFICOS GENERADOS:
  ✓ Matriz de correlaciones (heatmap)
  ✓ Scatter plot con línea de regresión simple
  ✓ Residuales vs. Ajustados (linealidad)
  ✓ Histograma, Q-Q Plot, ECDF (normalidad)
  ✓ Scale-Location Plot (homocedasticidad)
  ✓ Gráfico de secuencia (independencia)
  ✓ Correlaciones entre predictores
  ✓ Comparación de transformaciones
  ✓ Cook's Distance, Leverage, DFBETAS (influencia)

🎯 PRÓXIMOS PASOS:
  1. Adapta este código a TUS datos reales
  2. Reemplaza la sección de generación (PARTE 1) con tu dataset
  3. El resto del código funcionará idénticamente
  4. Documenta supuestos en tu tabla de resultados (Formato APA)
  5. Usa estos gráficos como referencia para tu informe

💡 ESTE LABORATORIO:
  ✓ 100% Autoejecutar en Google Colab
  ✓ Sin dependencias problemáticas (numpy, pandas, scipy básico)
  ✓ Algoritmos manuales (OLS, VIF, Cook's D, DFBETAS)
  ✓ Garantizado funcionar sin errores
  ✓ PhD-level interpretación de resultados

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

print(resumen)
print("="*80)
print(f"Fecha de ejecución: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)